In [ ]:
import pandas as pd, random
INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"

def leer_fasta(path):
    seqs, h, ch = {}, None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if h: seqs[h] = "".join(ch)
                h = line[1:].split()[0]; ch = []
            else: ch.append(line.strip())
        if h: seqs[h] = "".join(ch)
    return seqs

pos = leer_fasta(f"{INTERIM}/combinado_v4_reducido.fasta")   # genes ARG completos
neg = leer_fasta(f"{INTERIM}/grupo_negativo_v5.fasta")       # ORFs negativos completos

random.seed(42)
pos_ids = random.sample(list(pos.keys()), 1000)
neg_ids = random.sample(list(neg.keys()), 1000)

with open(f"{INTERIM}/muestra_genes_completos.fasta", "w") as f:
    for gid in pos_ids: f.write(f">{gid}\n{pos[gid]}\n")
    for gid in neg_ids: f.write(f">{gid}\n{neg[gid]}\n")

with open(f"{INTERIM}/muestra_genes_labels.tsv", "w") as f:
    f.write("id\tetiqueta\n")
    for gid in pos_ids: f.write(f"{gid}\t1\n")
    for gid in neg_ids: f.write(f"{gid}\t0\n")

print(f"Genes completos: {len(pos_ids)} positivos + {len(neg_ids)} negativos")

Lecturas de 150 nt generadas: 11392


In [ ]:
import numpy as np, pandas as pd
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"
ARGNET_GENES = "/home/r4gbi/ARGNet/results/comparacion_genes_argnet.txt"

# 1) out-of-fold por fragmento 
df = pd.read_csv(f"{INTERIM}/features_kmer4_fragmentado.csv")
df["gen"] = df["id"].str.split("__frag").str[0]
X = df.drop(columns=["id","etiqueta","gen"]).values; y = df["etiqueta"].values; g = df["gen"].values
oof = np.full(len(y), -1)
skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
rng = np.random.default_rng(42)
for tr, va in skf.split(X, y, g):
    p = np.where(y[tr]==1)[0]; n = np.where(y[tr]==0)[0]
    bal = np.concatenate([p, rng.choice(n, size=len(p), replace=False)])
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(X[tr][bal], y[tr][bal]); oof[va] = rf.predict(X[va])

# 2) agregar a nivel de gen: gen = ARG si ALGUN fragmento es ARG
por_gen = defaultdict(list)
for gid, pr in zip(df["gen"], oof): por_gen[gid].append(pr)
modelo_gen = {gid: (1 if any(v) else 0) for gid, v in por_gen.items()}

# 3) verdad + ARGNet
labels = pd.read_csv(f"{INTERIM}/muestra_genes_labels.tsv", sep="\t")
verdad = dict(zip(labels["id"], labels["etiqueta"]))
argnet_gen = {}
with open(ARGNET_GENES) as f:
    next(f)
    for line in f:
        p = line.rstrip("\n").split("\t")
        argnet_gen[p[0]] = 1 if p[1] == "ARG" else 0

ids = [i for i in labels["id"] if i in argnet_gen and i in modelo_gen]
yt = [verdad[i] for i in ids]; ym = [modelo_gen[i] for i in ids]; ya = [argnet_gen[i] for i in ids]

def rep(nombre, yt, yp):
    cm = confusion_matrix(yt, yp, labels=[0,1])
    print(f"\n=== {nombre} ===")
    print(f"  Real NEG   {cm[0,0]:>4}   {cm[0,1]:>4}")
    print(f"  Real POS   {cm[1,0]:>4}   {cm[1,1]:>4}")
    print(f"  Recall={recall_score(yt,yp):.3f}  Precision={precision_score(yt,yp,zero_division=0):.3f}  F1={f1_score(yt,yp):.3f}")

print(f"Genes comparados: {len(ids)}")
rep("ARGus (fragmenta + agrega)", yt, ym)
rep("ARGNet (gen completo)", yt, ya)

/tmp/ipykernel_2264/3504052559.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["gen"] = df["id"].str.split("__frag").str[0]


Genes comparados: 2000

=== ARGus (fragmenta + agrega) ===
  Real NEG    695    305
  Real POS    107    893
  Recall=0.893  Precision=0.745  F1=0.813

=== ARGNet (gen completo) ===
  Real NEG    912     88
  Real POS    447    553
  Recall=0.553  Precision=0.863  F1=0.674


In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"
labels = pd.read_csv(f"{INTERIM}/muestra_genes_labels.tsv", sep="\t")
verdad = dict(zip(labels["id"], labels["etiqueta"]))

# --- ARGNet (columna 2 = ARG / non-ARG) ---
argnet = {}
with open("/home/r4gbi/ARGNet/results/comparacion_genes_argnet.txt") as f:
    next(f)
    for line in f:
        p = line.rstrip("\n").split("\t")
        argnet[p[0]] = 1 if p[1] == "ARG" else 0

# --- DeepARG (read_id presente en el archivo = ARG) ---
deeparg_pos = set()
with open(f"{INTERIM}/deeparg_genes.mapping.ARG") as f:
    for line in f:
        if line.startswith("#"): continue
        c = line.rstrip("\n").split("\t")
        if len(c) > 3: deeparg_pos.add(c[3])
deeparg = {gid: (1 if gid in deeparg_pos else 0) for gid in labels["id"]}

def reporte(nombre, pred):
    ids = [i for i in labels["id"] if i in pred]
    yt = [verdad[i] for i in ids]
    yp = [pred[i] for i in ids]
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    print(f"\n{'='*55}\n  {nombre}  (genes evaluados: {len(ids)})\n{'='*55}")
    print("Matriz de confusión:")
    print(f"                  Predijo NEG   Predijo POS")
    print(f"  Real NEG (0)      {cm[0,0]:>8}      {cm[0,1]:>8}")
    print(f"  Real POS (1)      {cm[1,0]:>8}      {cm[1,1]:>8}")
    print()
    print(classification_report(yt, yp, target_names=["no_resist (0)", "resist (1)"], digits=3))

reporte("ARGNet", argnet)
reporte("DeepARG", deeparg)


  ARGNet  (genes evaluados: 2000)
Matriz de confusión:
                  Predijo NEG   Predijo POS
  Real NEG (0)           912            88
  Real POS (1)           447           553

               precision    recall  f1-score   support

no_resist (0)      0.671     0.912     0.773      1000
   resist (1)      0.863     0.553     0.674      1000

     accuracy                          0.733      2000
    macro avg      0.767     0.733     0.724      2000
 weighted avg      0.767     0.733     0.724      2000


  DeepARG  (genes evaluados: 2000)
Matriz de confusión:
                  Predijo NEG   Predijo POS
  Real NEG (0)           998             2
  Real POS (1)           649           351

               precision    recall  f1-score   support

no_resist (0)      0.606     0.998     0.754      1000
   resist (1)      0.994     0.351     0.519      1000

     accuracy                          0.674      2000
    macro avg      0.800     0.674     0.636      2000
 weighted avg  

In [14]:
fallos_argnet = [i for i in ids if verdad[i]==1 and argnet_gen[i]==0]
from collections import Counter
fuentes = Counter(i.split("|")[0] for i in fallos_argnet)
print(f"ARGNet marcó como non-ARG {len(fallos_argnet)} genes que SÍ son ARG.")
print("Por base de datos de origen:")
for fuente, n in fuentes.most_common():
    print(f"  {fuente}: {n}")

ARGNet marcó como non-ARG 447 genes que SÍ son ARG.
Por base de datos de origen:
  ResFinderFG: 263
  CARD: 111
  MEGARes: 40
  AMRFinderPlus: 19
  ResFinder: 11
  ARGANNOT: 3
